In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras import layers, models, losses 
from tensorflow.keras.datasets import mnist 
from tensorflow.keras.metrics import Precision, Recall
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from scipy.ndimage import rotate

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
# imgs
print(x_train.shape == (60000, 28, 28))
print(x_test.shape == (10000, 28, 28))
# labels 
print(y_train.shape == (60000,))
print(y_test.shape == (10000,))

True
True
True
True


In [3]:
angles = [i for i in range(-20, 21, 5)]
NOISE_STD = 20.0
new_x = []
new_y = []

for img, label in zip(x_train, y_train):
    # without augmentation
    new_x.append(img)
    new_y.append(label)

    for angle in angles:
        if angle == 0:
            noise = np.random.normal(loc=0.0, scale=NOISE_STD, size=img.shape)
            img_noised = img + noise
            img_noised = np.clip(img_noised, 0, 255).astype(np.uint8)
            new_x.append(img_noised)
            new_y.append(label)
            continue

        img_rot = rotate(img, angle, axes=(0,1), reshape=False, mode='constant', cval=0.0)
        new_x.append(img_rot)
        new_y.append(label)
        noise = np.random.normal(loc=0.0, scale=NOISE_STD, size=img_rot.shape)
        img_noised = img_rot + noise
        img_noised = np.clip(img_noised, 0, 255).astype(np.uint8)

        new_x.append(img_noised)
        new_y.append(label)

x_train_augmented = np.array(new_x)
y_train_augmented = np.array(new_y)


In [4]:
model = models.Sequential([
    layers.Input((28,28,1)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(pool_size=(2)),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(pool_size=(2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

In [5]:
model.compile(optimizer='adam', loss=losses.SparseCategoricalCrossentropy(), metrics=['accuracy'],)
model.fit(x=x_train_augmented, y=y_train_augmented, batch_size=64, epochs=1, validation_data=(x_test, y_test))

16875/16875 ━━━━━━━━━━━━━━━━━━━━ 57s 3ms/step - accuracy: 0.9796 - loss: 0.0666 - val_accuracy: 0.9922 - val_loss: 0.0265


In [6]:
model_loss, model_accuracy = model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9922 - loss: 0.0265


In [7]:
print(f"Loss: {model_loss}")
print(f"Accuracy: {model_accuracy*100:.2f}")

Loss: 0.0265476256608963
Accuracy: 99.22


In [8]:
predictions = model.predict(x_test)
decisions = np.argmax(predictions, axis=1)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [9]:
print(classification_report(y_test, decisions))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00       980
           1       1.00      0.99      1.00      1135
           2       1.00      0.99      1.00      1032
           3       0.99      1.00      0.99      1010
           4       0.99      0.99      0.99       982
           5       0.99      0.98      0.99       892
           6       0.99      0.99      0.99       958
           7       0.98      1.00      0.99      1028
           8       1.00      0.99      0.99       974
           9       0.99      0.99      0.99      1009

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000



In [10]:
cm = confusion_matrix(y_test, decisions)
df_cm = pd.DataFrame(cm)
df_cm.index.name = "Real"
df_cm.columns.name = "Predicted"
df_cm

Predicted,0,1,2,3,4,5,6,7,8,9
Real,,,,,,,,,,
0,975,1,0,0,1,0,1,2,0,0
1,0,1129,0,0,0,0,2,4,0,0
2,0,0,1026,1,1,0,0,4,0,0
3,0,0,0,1008,0,1,0,0,1,0
4,0,0,0,0,975,0,0,1,0,6
5,1,0,0,8,0,878,2,2,0,1
6,2,2,0,0,1,2,950,0,1,0
7,0,2,2,0,0,0,0,1023,0,1
8,0,0,2,2,1,1,0,1,964,3


In [11]:
model.save('../models/digit-recognizer-with-augmentation.keras')